# ERA5 Dataset Exploration

**Topics covered:**
1. Pressure levels — what they are, which ones matter, how to slice
2. Geopotential height — Z500 as a circulation diagnostic
3. Wind vectors — U/V components, wind speed, visualization
4. Humidity variables — specific humidity, relative humidity, VPD

**Data source:** ERA5 via Pangeo zarr store (no API key required)  
**Environment:** Compatible with existing ERA5 pipeline setup

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import zarr
import gcsfs

# Optional but recommended
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

---
## Load ERA5 from Pangeo

The Pangeo ERA5 zarr store contains the full ERA5 reanalysis at 0.25° resolution.  
Opens it lazily, e.g. nothing is downloaded until actual compute on a slice.

In [ ]:
# Open ERA5 pressure-level data from Pangeo (lazy, no download yet)
fs = gcsfs.GCSFileSystem(token='anon')

# Pressure-level variables (geopotential, wind, humidity all live here)
mapper_pl = fs.get_mapper('gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr')
ds = xr.open_zarr(mapper_pl, consolidated=True)

print(ds)
print('\nVariables available:')
for v in sorted(ds.data_vars):
    print(f'  {v:40s}  {ds[v].dims}')

In [ ]:
# Select a single time slice for exploration — June 15 2020, 00 UTC
# This is our working snapshot throughout the notebook
TIME = '2024-06-15T00:00'

snap = ds.sel(time=TIME, method='nearest')
print(f'Snapshot time: {snap.time.values}')
print(f'Latitude range: {float(snap.latitude.min()):.1f} to {float(snap.latitude.max()):.1f}')
print(f'Longitude range: {float(snap.longitude.min()):.1f} to {float(snap.longitude.max()):.1f}')

---
## 1. Pressure Levels

### What are pressure levels?

The atmosphere is a 3D fluid. ERA5 samples it on **isobaric surfaces** — surfaces of constant pressure —
rather than constant altitude. Pressure decreases with altitude, so higher pressure = closer to the surface.

```
~250 hPa  →  ~10 km   (upper troposphere, jet stream level)
~500 hPa  →  ~5.5 km  (mid-troposphere, synoptic steering level)
~850 hPa  →  ~1.5 km  (lower troposphere, air mass boundaries)
~1000 hPa →  ~0.1 km  (near surface)
```

### Why not just use altitude?

Pressure coordinates are **terrain-following** in a useful way: a given pressure level cuts through
the atmosphere at roughly the same density everywhere, making horizontal comparisons physically
meaningful. Altitude coordinates intersect mountains and don't have that property.

### Which levels matter for ML weather models?

| Level | Physical meaning | Used for |
|-------|-----------------|----------|
| 250 hPa | Jet stream, divergence | Upper-level dynamics |
| 500 hPa | Mid-level steering flow | Z500 geopotential — the canonical circulation variable |
| 700 hPa | Mid-troposphere moisture | Precipitation diagnostics |
| 850 hPa | Lower troposphere | T850 — air mass boundaries, fronts |
| 1000 hPa | Near surface | Boundary layer variables |

In [ ]:
# Inspect available pressure levels in the dataset
if 'level' in ds.dims:
    levels = ds.level.values
    print(f'Pressure levels available ({len(levels)} total):')
    print(levels)
elif 'pressure_level' in ds.dims:
    levels = ds.pressure_level.values
    print(f'Pressure levels available ({len(levels)} total):')
    print(levels)

In [ ]:
# Vertical profile: temperature at a single grid point (Chicago: 41.8N, 272E)
# This shows how temperature varies with pressure level

level_dim = 'level' if 'level' in ds.dims else 'pressure_level'

t_profile = snap['temperature'].sel(latitude=41.8, longitude=272.0, method='nearest').compute()

fig, ax = plt.subplots(figsize=(5, 7))
ax.plot(t_profile.values - 273.15, t_profile[level_dim].values, 'o-', color='#2E6DA4', linewidth=2, markersize=6)
ax.invert_yaxis()  # Pressure decreases upward
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Pressure (hPa)', fontsize=12)
ax.set_title('Vertical temperature profile\nChicago, 2020-01-15 00 UTC', fontsize=12)
ax.axhline(500, color='#f59e0b', linestyle='--', alpha=0.7, label='500 hPa (Z500 level)')
ax.axhline(850, color='#ef4444', linestyle='--', alpha=0.7, label='850 hPa (T850 level)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

t850 = float(t_profile.sel({level_dim: 850}, method='nearest').values) - 273.15
t500 = float(t_profile.sel({level_dim: 500}, method='nearest').values) - 273.15
print(f'1T850: {t850:.1f}°C  (air mass boundary diagnostic)')
print(f'T500: {t500:.1f}°C  (mid-level lapse rate diagnostic)')

---
## 2. Geopotential Height (Z500)

### What is geopotential height?

**Geopotential (Z)** is the gravitational potential energy per unit mass at a given point.
**Geopotential height** converts this to meters: `Z_height = Z / g₀` where g₀ = 9.80665 m/s².

At 500 hPa, geopotential height tells you **how high up you have to go to reach 500 hPa pressure**.
Where the atmosphere is warm and expanded, 500 hPa is higher up. Where it is cold and dense, 500 hPa
is lower.

### Why Z500 is the canonical circulation variable

Z500 is the **fingerprint of the large-scale wave pattern** that organizes weather at the synoptic scale.

```
High Z500 (ridge)  →  warm, sinking air, fair weather, anticyclonic flow
Low Z500 (trough)  →  cold, rising air, storms, cyclonic flow
```

The gradient of Z500 drives the geostrophic wind — the large-scale steering flow that moves
surface weather systems around. If you understand the Z500 pattern, you understand where
storms will go.

This is why GraphCast and FourCastNet both use Z500 as a primary variable and why
it's a standard verification target.

In [ ]:
# Extract Z500 — geopotential at 500 hPa, convert to geopotential height
g0 = 9.80665  # standard gravity m/s²

level_dim = 'level' if 'level' in ds.dims else 'pressure_level'

z500 = snap['geopotential'].sel({level_dim: 500}, method='nearest').compute() / g0
print(f'Z500 shape: {z500.shape}')
print(f'Z500 range: {float(z500.min()):.0f} to {float(z500.max()):.0f} m')
print(f'Z500 mean:  {float(z500.mean()):.0f} m  (should be ~5500m)')

In [ ]:
# Plot Z500 — Northern Hemisphere, showing the wave pattern
fig = plt.figure(figsize=(12, 7))
ax = plt.axes(projection=ccrs.NorthPolarStereo())
ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())

# Filled contours
cf = ax.contourf(
    z500.longitude, z500.latitude, z500.values,
    levels=20, cmap='RdBu_r',
    transform=ccrs.PlateCarree()
)

# Contour lines — these ARE the wave pattern
cs = ax.contour(
    z500.longitude, z500.latitude, z500.values,
    levels=15, colors='k', linewidths=0.5, alpha=0.5,
    transform=ccrs.PlateCarree()
)
ax.clabel(cs, inline=True, fontsize=7, fmt='%d')

ax.add_feature(cfeature.COASTLINE, linewidth=0.8, color='white')
ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5, color='white')
ax.gridlines(draw_labels=False, linewidth=0.5, alpha=0.3)

plt.colorbar(cf, ax=ax, shrink=0.7, label='Geopotential height (m)')
ax.set_title('Z500 — Geopotential height at 500 hPa\n2020-01-15 00 UTC', fontsize=13)
plt.tight_layout()
plt.show()

print('Ridges (warm colors) = high Z500 = warm/sinking air = fair weather')
print('Troughs (cool colors) = low Z500 = cold/rising air = storm tracks')

In [ ]:
# Z500 anomaly — deviation from the climatological mean
# This is what ML models actually need to predict well
# (the mean is easy; the anomaly is the weather)

# Simple anomaly: subtract spatial mean for this snapshot
# In practice you'd subtract a climatological mean computed over 30+ years
z500_nh = z500.sel(latitude=slice(90, 20))  # Northern Hemisphere
z500_anomaly = z500_nh - float(z500_nh.mean())

fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.NorthPolarStereo())
ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())

levels_anom = np.linspace(-400, 400, 21)
cf = ax.contourf(
    z500_anomaly.longitude, z500_anomaly.latitude, z500_anomaly.values,
    levels=levels_anom, cmap='RdBu_r', extend='both',
    transform=ccrs.PlateCarree()
)
ax.add_feature(cfeature.COASTLINE, linewidth=0.8, color='white')
ax.gridlines(draw_labels=False, linewidth=0.5, alpha=0.3)
plt.colorbar(cf, ax=ax, shrink=0.7, label='Z500 anomaly (m)')
ax.set_title('Z500 anomaly (deviation from snapshot mean)\n2020-01-15 00 UTC', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Wind Vectors

### U and V components

Wind is a vector quantity. ERA5 stores it as two scalar fields:

```
U  →  zonal wind component      (positive = westerly, blowing east)
V  →  meridional wind component (positive = southerly, blowing north)
```

Wind speed = √(U² + V²)  
Wind direction = atan2(U, V)

### Wind matters for your background

Your 2010 simulation included **wind disturbance regimes** — the probability of stand-replacing
wind events driving forest compositional change. That required knowing not just mean wind speed
but the **upper tail of the wind speed distribution** and how it varies spatially with terrain.

At the synoptic scale, the **250 hPa jet stream** (U250) is the dominant wind feature —
it steers storm tracks and controls the speed at which weather systems move.

In [ ]:
# Extract U and V at 850 hPa and 250 hPa
u850 = snap['u_component_of_wind'].sel({level_dim: 850}, method='nearest').compute()
v850 = snap['v_component_of_wind'].sel({level_dim: 850}, method='nearest').compute()
u250 = snap['u_component_of_wind'].sel({level_dim: 250}, method='nearest').compute()
v250 = snap['v_component_of_wind'].sel({level_dim: 250}, method='nearest').compute()

# Compute wind speed
wspd850 = np.sqrt(u850**2 + v850**2)
wspd250 = np.sqrt(u250**2 + v250**2)

print(f'850 hPa wind speed: mean={float(wspd850.mean()):.1f} m/s, max={float(wspd850.max()):.1f} m/s')
print(f'250 hPa wind speed: mean={float(wspd250.mean()):.1f} m/s, max={float(wspd250.max()):.1f} m/s')
print(f'\nJet stream (250 hPa max) at: {float(wspd250.max()):.1f} m/s = {float(wspd250.max())*1.944:.0f} knots')

In [ ]:
# Plot 250 hPa jet stream
fig = plt.figure(figsize=(14, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()

cf = ax.contourf(
    wspd250.longitude, wspd250.latitude, wspd250.values,
    levels=np.arange(0, 80, 5), cmap='plasma',
    transform=ccrs.PlateCarree()
)

# Subsample wind vectors for quiver plot (every 10th point)
step = 10
lons = u250.longitude.values[::step]
lats = u250.latitude.values[::step]
U = u250.values[::step, ::step]
V = v250.values[::step, ::step]

ax.quiver(lons, lats, U, V, transform=ccrs.PlateCarree(),
          scale=800, width=0.002, alpha=0.6, color='white')

ax.add_feature(cfeature.COASTLINE, linewidth=0.8, color='white')
ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5, color='white')
gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.3)
gl.top_labels = False
gl.right_labels = False

plt.colorbar(cf, ax=ax, shrink=0.8, label='Wind speed (m/s)')
ax.set_title('250 hPa wind speed + vectors (jet stream)\n2020-01-15 00 UTC', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 850 hPa wind — lower troposphere, more directly tied to surface weather
# This is where frontal boundaries show up as wind shifts

# Focus on North America
u850_na = u850.sel(latitude=slice(70, 20), longitude=slice(220, 310))
v850_na = v850.sel(latitude=slice(70, 20), longitude=slice(220, 310))
wspd850_na = np.sqrt(u850_na**2 + v850_na**2)

fig = plt.figure(figsize=(10, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-140, -50, 20, 70], crs=ccrs.PlateCarree())

cf = ax.contourf(
    wspd850_na.longitude, wspd850_na.latitude, wspd850_na.values,
    levels=np.arange(0, 35, 2.5),  cmap="RdBu_r",
    transform=ccrs.PlateCarree()
)

step = 4
ax.quiver(
    u850_na.longitude.values[::step], u850_na.latitude.values[::step],
    u850_na.values[::step, ::step], v850_na.values[::step, ::step],
    transform=ccrs.PlateCarree(), scale=250, width=0.003, alpha=0.8
)

ax.add_feature(cfeature.COASTLINE, linewidth=2, color='white')
ax.add_feature(cfeature.BORDERS, linewidth=1, color='white')
ax.add_feature(cfeature.STATES, linewidth=0.3, alpha=1, color='white')
gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.3)
gl.top_labels = False
gl.right_labels = False

plt.colorbar(cf, ax=ax, shrink=0.8, label='Wind speed (m/s)')
ax.set_title('850 hPa wind — North America\n2020-01-15 00 UTC\nWind shifts here mark frontal boundaries', fontsize=11)
plt.tight_layout()
plt.show()

---
## 4. Humidity Variables

### The variables

ERA5 provides several humidity-related fields:

| Variable | Units | Physical meaning |
|----------|-------|------------------|
| `specific_humidity` (q) | kg/kg | Mass of water vapor per unit mass of moist air |
| `relative_humidity` (RH) | % | Actual humidity / saturation humidity |
| `2m_dewpoint_temperature` (Td) | K | Temperature at which air becomes saturated |
| `2m_temperature` (T2m) | K | Near-surface air temperature |

### VPD — the variable that connects to your 2006 work

**Vapor Pressure Deficit (VPD)** is the difference between the saturation vapor pressure
and the actual vapor pressure. It is the primary driver of **atmospheric demand for evapotranspiration**
and of **fuel moisture dynamics** in fire-prone ecosystems.

```
High VPD  →  dry atmosphere aggressively pulling moisture from vegetation
           →  live fuel moisture falls
           →  dead fuel moisture responds within hours
           →  fire risk rises
```

This is the direct link between your Penman-Monteith ET model (which is VPD-driven)
and the ERA5 humidity fields you'd use in a modern ML model.

In [ ]:
# Load surface variables for VPD calculation
# VPD requires T2m and dewpoint (or specific humidity)

# Try to get single-level variables
# WeatherBench2 stores these differently — check what's available
print('Checking for surface variables...')
surface_vars = [v for v in ds.data_vars if '2m' in v or 'surface' in v.lower() or 'dewpoint' in v.lower()]
print('Surface-related variables found:')
for v in surface_vars:
    print(f'  {v}')

# Also check specific humidity at 850 hPa as a proxy for lower troposphere moisture
if 'specific_humidity' in ds.data_vars:
    q850 = snap['specific_humidity'].sel({level_dim: 850}, method='nearest').compute()
    print(f'\nq850 range: {float(q850.min()*1000):.2f} to {float(q850.max()*1000):.2f} g/kg')

In [ ]:
# Compute VPD from T2m and dewpoint temperature
# Both should be available in WeatherBench2

def saturation_vapor_pressure(T_kelvin):
    """
    Tetens formula for saturation vapor pressure.
    T in Kelvin, returns pressure in hPa.
    This is the same formula used in Penman-Monteith ET.
    """
    T_c = T_kelvin - 273.15
    return 6.1078 * np.exp(17.27 * T_c / (T_c + 237.3))

def compute_vpd(T2m_K, Td_K):
    """
    VPD = saturation vapor pressure - actual vapor pressure
    Actual vapor pressure = saturation vapor pressure at dewpoint
    Units: hPa
    """
    es = saturation_vapor_pressure(T2m_K)   # saturation at air temp
    ea = saturation_vapor_pressure(Td_K)    # actual (= sat at dewpoint)
    return es - ea

# Load T2m and dewpoint
try:
    t2m = snap['2m_temperature'].compute()
    td2m = snap['2m_dewpoint_temperature'].compute()
    
    vpd = compute_vpd(t2m.values, td2m.values)
    print(f'VPD computed successfully')
    print(f'VPD range: {vpd.min():.2f} to {vpd.max():.2f} hPa')
    print(f'VPD mean:  {vpd.mean():.2f} hPa')
    print(f'\nNote: VPD > 15 hPa = high fire risk in most ecosystems')
    print(f'      VPD > 25 hPa = extreme fire weather')
    
except KeyError as e:
    print(f'Variable not found: {e}')
    print('Trying alternative variable names...')
    print([v for v in ds.data_vars])

In [ ]:
def saturation_vapor_pressure(T_kelvin):
    T_c = T_kelvin - 273.15
    return 6.1078 * np.exp(17.27 * T_c / (T_c + 237.3))

def compute_vpd(T2m_K, Td_K):
    es = saturation_vapor_pressure(T2m_K)
    ea = saturation_vapor_pressure(Td_K)
    return es - ea

def q_to_dewpoint(q_kgkg, p_hpa=1013.25):
    ea = q_kgkg * p_hpa / (0.622 + 0.378 * q_kgkg)
    td_c = (237.3 * np.log(ea / 6.1078)) / (17.27 - np.log(ea / 6.1078))
    return td_c + 273.15

In [ ]:
print(f'VPD (850 hPa proxy) min:  {vpd_850.min():.2f} hPa')
print(f'VPD (850 hPa proxy) max:  {vpd_850.max():.2f} hPa')
print(f'VPD (850 hPa proxy) mean: {vpd_850.mean():.2f} hPa')

# Plot
fig = plt.figure(figsize=(14, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()

cf = ax.contourf(
    q850_snap.longitude, q850_snap.latitude, vpd_850,
    levels=np.arange(0, 25, 1.5), cmap='YlOrRd', extend='max',
    transform=ccrs.PlateCarree()
)
ax.contour(
    q850_snap.longitude, q850_snap.latitude, vpd_850,
    levels=[10], colors='darkred', linewidths=1.5,
    transform=ccrs.PlateCarree()
)
ax.add_feature(cfeature.COASTLINE, linewidth=0.8, color='white')
ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5, color='white')
gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.3)
gl.top_labels = False
gl.right_labels = False

plt.colorbar(cf, ax=ax, shrink=0.8, label='VPD proxy (hPa)')
ax.set_title('VPD proxy from 850 hPa q and T\n2020-01-15 00 UTC', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
print(f'VPD min: {vpd.min():.3f}')
print(f'VPD max: {vpd.max():.3f}')
print(f'VPD mean: {vpd.mean():.3f}')
print(f't2m shape: {t2m.shape}')
print(f'td2m shape: {td2m.shape}')
print(f'Sample T2m value: {float(t2m.isel(latitude=100, longitude=100).values):.1f} K')
print(f'Sample Td value: {float(td2m.isel(latitude=100, longitude=100).values):.1f} K')

In [ ]:
# Derive dewpoint from specific humidity at surface pressure
# q is specific humidity in kg/kg, p in hPa
# Magnus formula inversion

def q_to_dewpoint(q_kgkg, p_hpa=1013.25):
    """
    Convert specific humidity to dewpoint temperature (Kelvin).
    q: specific humidity kg/kg
    p: surface pressure hPa
    """
    # Actual vapor pressure from specific humidity
    ea = q_kgkg * p_hpa / (0.622 + 0.378 * q_kgkg)
    # Invert Magnus formula for dewpoint
    td_c = (237.3 * np.log(ea / 6.1078)) / (17.27 - np.log(ea / 6.1078))
    return td_c + 273.15

# Get q at 850 hPa as proxy (or check if surface q exists)
print([v for v in ds.data_vars if 'humid' in v.lower() or v.startswith('q')])

In [ ]:
# Specific humidity vertical profile — how moisture decreases with altitude
# This matters for precipitation physics and convective instability

if 'specific_humidity' in ds.data_vars:
    q_profile = snap['specific_humidity'].sel(
        latitude=41.8, longitude=272.0, method='nearest'
    ).compute()

    fig, ax = plt.subplots(figsize=(5, 7))
    ax.plot(q_profile.values * 1000, q_profile[level_dim].values,
            'o-', color='#06b6d4', linewidth=2, markersize=6)
    ax.invert_yaxis()
    ax.set_xlabel('Specific humidity (g/kg)', fontsize=12)
    ax.set_ylabel('Pressure (hPa)', fontsize=12)
    ax.set_title('Specific humidity profile\nChicago, 2020-01-15 00 UTC', fontsize=12)
    ax.axhline(850, color='#ef4444', linestyle='--', alpha=0.7, label='850 hPa')
    ax.axhline(500, color='#f59e0b', linestyle='--', alpha=0.7, label='500 hPa')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    q850_val = float(q_profile.sel({level_dim: 850}, method='nearest').values) * 1000
    q500_val = float(q_profile.sel({level_dim: 500}, method='nearest').values) * 1000
    print(f'q850: {q850_val:.2f} g/kg  (lower troposphere moisture)')
    print(f'q500: {q500_val:.2f} g/kg  (mid-level moisture — much drier)')

---
## Summary: Variables used in ML weather models

The table below maps ERA5 variable names to their physical role and relevance
to your background.

| ERA5 variable | Level | Physical role | Your 2006 connection |
|---------------|-------|---------------|----------------------|
| `geopotential` → Z500 | 500 hPa | Large-scale circulation, storm steering | Synoptic forcing that drove GCM scenarios |
| `temperature` → T850 | 850 hPa | Air mass boundaries, frontal zones | Temperature forcing for land surface model |
| `u_component_of_wind`, `v_component_of_wind` | 250 hPa | Jet stream | Wind disturbance regime input |
| `specific_humidity` → q850 | 850 hPa | Lower troposphere moisture | Penman-Monteith atmospheric demand |
| `2m_temperature` | Surface | Near-surface temp | PRISM-calibrated downscaled forcing |
| `2m_dewpoint_temperature` | Surface | Surface moisture | VPD calculation → fuel moisture |
| VPD (derived) | Surface | Atmospheric ET demand, fire risk | Direct input to your fuel moisture model |

### The key physical insight 

These are not feature names in a config file. They form a physically coupled system:

```
Z500 wave pattern
    ↓ steers storm tracks
T850 air mass boundaries
    ↓ control surface temperature
T2m + Td → VPD
    ↓ drives atmospheric demand
ET demand → soil moisture → ANPP → fuel moisture → fire risk
```

This is the chain you modeled in 2006. ERA5 provides the top of it.

In [ ]:
# Quick reference: how to pull any variable for your training pipeline

def get_era5_variable(ds, variable, level=None, level_dim='level',
                      lat_slice=slice(90, -90), lon_slice=slice(0, 360)):
    """
    Convenience function: extract a variable from ERA5 with optional
    pressure level selection and spatial subsetting.
    
    Returns an xarray DataArray, still lazy (no compute).
    """
    da = ds[variable].sel(latitude=lat_slice, longitude=lon_slice)
    if level is not None:
        da = da.sel({level_dim: level}, method='nearest')
    return da

# Example usage
level_dim = 'level' if 'level' in ds.dims else 'pressure_level'

z500_lazy = get_era5_variable(ds, 'geopotential', level=500, level_dim=level_dim)
t850_lazy = get_era5_variable(ds, 'temperature', level=850, level_dim=level_dim)
u250_lazy = get_era5_variable(ds, 'u_component_of_wind', level=250, level_dim=level_dim)

print('Variable shapes (lazy):')
print(f'  Z500: {z500_lazy.shape}')
print(f'  T850: {t850_lazy.shape}')
print(f'  U250: {u250_lazy.shape}')
print('\nNone of this has been downloaded yet.')
print('Call .compute() or .sel(time=...) to trigger actual data loading.')